In [1]:
# pip install control matplotlib ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import control as ct

from ipywidgets import interact, FloatSlider, fixed


# ==============================================================================
# CONFIG
# ==============================================================================
G = 9.80665

KP0 = 0.44
KD0 = 0.155
H_REF0 = 5.0

K_MAX0 = 6.0


# ==============================================================================
# MODEL
# ==============================================================================
def make_pd_loop(kp, kd, h_ref):
    """
    Simplified lateral model:
        xddot = Kx * u
        u = -(kp*x + kd*xdot)

    For classical unity feedback form:
        P(s) = Kx / s^2
        C(s) = kd*s + kp
        L(s) = C(s)P(s)
        T(s) = L(s)/(1+L(s))
    """
    if h_ref <= 0:
        raise ValueError("h_ref must be positive.")

    Kx = G / h_ref

    s = ct.tf([1, 0], [1])   # s
    P = Kx / s**2
    C = kd * s + kp
    L = C * P
    T = ct.feedback(L, 1)    # negative feedback by default

    return P, C, L, T, Kx


def second_order_metrics(kp, kd, h_ref):
    Kx = G / h_ref

    if kp <= 0:
        return np.nan, np.nan

    wn = np.sqrt(Kx * kp)
    zeta = 0.5 * kd * np.sqrt(Kx / kp)

    return wn, zeta


def get_root_locus_loci(L, gains):
    """
    Uses modern python-control root_locus_map.
    Falls back to older root_locus syntax if needed.
    """
    if hasattr(ct, "root_locus_map"):
        rldata = ct.root_locus_map(L, gains=gains)
        return np.asarray(rldata.loci), np.asarray(rldata.gains)

    # Fallback for older versions
    try:
        roots, gains_out = ct.root_locus(L, gains=gains, plot=False)
    except TypeError:
        roots, gains_out = ct.root_locus(L, kvect=gains, Plot=False)

    return np.asarray(roots), np.asarray(gains_out)


# ==============================================================================
# PLOTTING
# ==============================================================================
def plot_explorer(kp=KP0, kd=KD0, h_ref=H_REF0, k_max=K_MAX0, t_end=15.0):
    P, C, L, T, Kx = make_pd_loop(kp, kd, h_ref)

    poles = ct.poles(T)
    wn, zeta = second_order_metrics(kp, kd, h_ref)

    gains = np.linspace(0.0, k_max, 1000)
    loci, gains_out = get_root_locus_loci(L, gains)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    # --------------------------------------------------------------------------
    # Root locus: scalar K multiplying the current PD regulator
    # --------------------------------------------------------------------------
    ax = axes[0]

    for branch in range(loci.shape[1]):
        ax.plot(loci[:, branch].real, loci[:, branch].imag, linewidth=1.5)

    ax.scatter(
        poles.real,
        poles.imag,
        marker="x",
        s=90,
        linewidths=2.5,
        label="current closed-loop poles",
    )

    ax.axvline(0.0, linewidth=1.2, alpha=0.7)
    ax.axhline(0.0, linewidth=0.8, alpha=0.7)

    ax.set_title("Classical root locus: K scales current PD")
    ax.set_xlabel("Re(s) [1/s]")
    ax.set_ylabel("Im(s) [1/s]")
    ax.grid(True, alpha=0.3)
    ax.legend()

    # --------------------------------------------------------------------------
    # Step response of closed-loop transfer function
    # --------------------------------------------------------------------------
    ax = axes[1]

    t = np.linspace(0.0, t_end, 800)
    tout, yout = ct.step_response(T, T=t)

    ax.plot(tout, yout, linewidth=1.8)
    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.7)

    ax.set_title("Closed-loop step response")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Normalized offset response")
    ax.grid(True, alpha=0.3)

    stable = np.all(np.real(poles) < 0)

    fig.suptitle(
        (
            f"kp={kp:.4f}, kd={kd:.4f}, H_ref={h_ref:.2f} m, "
            f"Kx={Kx:.3f} 1/s²/rad | "
            f"wn={wn:.3f} rad/s, zeta={zeta:.3f}, stable={stable}"
        ),
        fontsize=10,
    )

    fig.tight_layout()
    plt.show()

    print("Closed-loop poles:")
    for p in poles:
        print(f"  {p.real:+.4f} {p.imag:+.4f}j")

    print()
    print(f"Kx   = {Kx:.4f} 1/s² per rad")
    print(f"wn   = {wn:.4f} rad/s  ({wn / (2*np.pi):.4f} Hz)")
    print(f"zeta = {zeta:.4f}")


# ==============================================================================
# INTERACTIVE NOTEBOOK SLIDERS
# ==============================================================================
interact(
    plot_explorer,
    kp=FloatSlider(value=KP0, min=0.001, max=1.5, step=0.01, description="kp"),
    kd=FloatSlider(value=KD0, min=0.0, max=0.8, step=0.005, description="kd"),
    h_ref=FloatSlider(value=H_REF0, min=1.0, max=10.0, step=0.25, description="H_ref [m]"),
    k_max=FloatSlider(value=K_MAX0, min=1.0, max=15.0, step=0.5, description="K max"),
    t_end=fixed(15.0),
);

interactive(children=(FloatSlider(value=0.44, description='kp', max=1.5, min=0.001, step=0.01), FloatSlider(va…